# Notebook 01 — Data Extraction
**NST DVA Capstone 2 | Healthcare Sector**

> **Purpose:** Load the raw diabetic patient dataset, perform an initial structural audit, and commit it untouched to `data/raw/`.
>
> ⚠️ **Rule:** This notebook must NEVER modify the source file. Only read and inspect.

---
**Pipeline Stage:** Extract → (Clean) → (Analyze) → (Load)

---

## 1. Environment Setup

In [ ]:
# ── Install / upgrade libraries if running in Colab ──────────────────────────
# Uncomment the line below only if running on Google Colab
# !pip install pandas numpy matplotlib seaborn --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded successfully.')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

## 2. Path Configuration

Update `RAW_DATA_PATH` to point to your raw dataset file.

In [ ]:
# ── Path Configuration ────────────────────────────────────────────────────────
# Update this path to match where you placed the raw dataset
RAW_DATA_PATH = '../data/raw/diabetic_data.csv'   # ← change filename if needed

# Verify the file exists before proceeding
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(
        f"❌ Raw dataset not found at: {RAW_DATA_PATH}\n"
        "   Please place the CSV in data/raw/ and update RAW_DATA_PATH above."
    )

print(f'✅ Dataset found: {RAW_DATA_PATH}')
print(f'   File size   : {os.path.getsize(RAW_DATA_PATH) / 1_000_000:.2f} MB')

## 3. Load Raw Dataset

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
# '?' is used as a missing-value marker in this dataset
df_raw = pd.read_csv(RAW_DATA_PATH, na_values=['?', 'Unknown', 'None', ''])

print('✅ Raw dataset loaded.')
print(f'   Rows    : {df_raw.shape[0]:,}')
print(f'   Columns : {df_raw.shape[1]}')

In [ ]:
# ── First look ────────────────────────────────────────────────────────────────
print('=== HEAD (first 5 rows) ===')
df_raw.head()

In [ ]:
print('=== TAIL (last 5 rows) ===')
df_raw.tail()

## 4. Schema & Data Types Audit

In [ ]:
# ── Column-level audit ────────────────────────────────────────────────────────
audit = pd.DataFrame({
    'dtype'         : df_raw.dtypes,
    'non_null_count': df_raw.count(),
    'null_count'    : df_raw.isnull().sum(),
    'null_pct'      : (df_raw.isnull().mean() * 100).round(2),
    'unique_values' : df_raw.nunique(),
    'sample_value'  : df_raw.iloc[0]
})

print('=== SCHEMA AUDIT ===')
audit

In [ ]:
# ── Capstone Gate Check ───────────────────────────────────────────────────────
MIN_ROWS = 5000
MIN_COLS = 8

rows_ok = df_raw.shape[0] >= MIN_ROWS
cols_ok = df_raw.shape[1] >= MIN_COLS

print('=== GATE 1 MINIMUM REQUIREMENTS ===')
print(f'   Row count  : {df_raw.shape[0]:,}  → {"✅ PASS" if rows_ok else "❌ FAIL"} (min {MIN_ROWS:,})')
print(f'   Column count: {df_raw.shape[1]}  → {"✅ PASS" if cols_ok else "❌ FAIL"} (min {MIN_COLS})')

if rows_ok and cols_ok:
    print('\n✅ Dataset meets minimum capstone requirements.')
else:
    print('\n⚠️  Dataset does NOT meet minimum requirements. Please review your dataset.')

## 5. Column Inventory & Classification

Classify each column by its analytical role so the cleaning notebook can target them correctly.

In [ ]:
# ── Identify numeric vs categorical columns ───────────────────────────────────
numeric_cols     = df_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_raw.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric columns    ({len(numeric_cols)})  : {numeric_cols}')
print()
print(f'Categorical columns ({len(categorical_cols)})  : {categorical_cols}')

In [ ]:
# ── Healthcare-specific column groups ─────────────────────────────────────────
# Adjust these lists if your dataset uses different column names

ID_COLS = ['encounter_id', 'patient_nbr']

DEMOGRAPHIC_COLS = ['race', 'gender', 'age', 'weight']

ADMISSION_COLS = [
    'admission_type_id', 'discharge_disposition_id',
    'admission_source_id', 'time_in_hospital',
    'payer_code', 'medical_specialty'
]

UTILIZATION_COLS = [
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient'
]

DIAGNOSIS_COLS = ['diag_1', 'diag_2', 'diag_3', 'number_diagnoses']

LAB_COLS = ['max_glu_serum', 'A1Cresult']

MEDICATION_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

CHANGE_COLS   = ['change', 'diabetesMed']
TARGET_COL    = ['readmitted']   # Outcome / target variable

print('✅ Column groups defined.')
print(f'   Medication columns tracked: {len(MEDICATION_COLS)}')

## 6. Missing Value Heatmap

In [ ]:
# ── Missing value summary ─────────────────────────────────────────────────────
missing = df_raw.isnull().mean().sort_values(ascending=False)
missing_cols = missing[missing > 0]

if len(missing_cols) == 0:
    print('✅ No missing values detected (after replacing ? with NaN).')
else:
    print(f'⚠️  {len(missing_cols)} columns have missing values:\n')
    print(missing_cols.to_string())

    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_cols) * 0.4)))
    missing_cols.plot(kind='barh', ax=ax, color='#E05C5C', edgecolor='white')
    ax.set_xlabel('Missing Value Proportion')
    ax.set_title('Missing Data by Column — Raw Dataset', fontsize=14, fontweight='bold')
    ax.axvline(0.3, color='orange', linestyle='--', linewidth=1.2, label='30% threshold')
    ax.axvline(0.5, color='red',    linestyle='--', linewidth=1.2, label='50% threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/raw/01_missing_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('   Chart saved to data/raw/01_missing_heatmap.png')

## 7. Duplicate Record Check

In [ ]:
# ── Full-row duplicates ───────────────────────────────────────────────────────
full_dupes = df_raw.duplicated().sum()
print(f'Full-row duplicates       : {full_dupes:,}')

# Duplicate encounter IDs (should be 0 for this dataset)
if 'encounter_id' in df_raw.columns:
    enc_dupes = df_raw['encounter_id'].duplicated().sum()
    print(f'Duplicate encounter_id    : {enc_dupes:,}')

# Patients with multiple encounters — clinically meaningful
if 'patient_nbr' in df_raw.columns:
    multi_enc = df_raw['patient_nbr'].duplicated().sum()
    unique_pts = df_raw['patient_nbr'].nunique()
    total_enc  = len(df_raw)
    print(f'Unique patients           : {unique_pts:,}  (out of {total_enc:,} encounters)')
    print(f'Patients with >1 encounter: {total_enc - unique_pts:,}')
    print('   ℹ️  Multiple encounters per patient are EXPECTED in this dataset.')

## 8. Target Variable Distribution

In [ ]:
# ── Readmission distribution ──────────────────────────────────────────────────
if 'readmitted' in df_raw.columns:
    readmit_counts = df_raw['readmitted'].value_counts()
    readmit_pct    = df_raw['readmitted'].value_counts(normalize=True) * 100

    print('=== READMISSION BREAKDOWN ===')
    for label in readmit_counts.index:
        print(f'   {label:>5}  →  {readmit_counts[label]:>7,}  ({readmit_pct[label]:.1f}%)')

    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#2196F3', '#FF5722', '#4CAF50']
    readmit_counts.plot(kind='bar', ax=ax, color=colors[:len(readmit_counts)],
                        edgecolor='white', rot=0)
    ax.set_title('Readmission Distribution — Raw Dataset', fontsize=14, fontweight='bold')
    ax.set_xlabel('Readmission Category')
    ax.set_ylabel('Number of Encounters')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig('../data/raw/01_readmission_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('   Chart saved to data/raw/01_readmission_distribution.png')

## 9. Basic Descriptive Statistics

In [ ]:
# ── Numeric summary ───────────────────────────────────────────────────────────
print('=== NUMERIC COLUMNS — DESCRIPTIVE STATISTICS ===')
df_raw[numeric_cols].describe().T

In [ ]:
# ── Categorical top-values ────────────────────────────────────────────────────
print('=== CATEGORICAL COLUMNS — TOP 5 VALUES ===')
for col in categorical_cols[:10]:  # limit to first 10 to keep output manageable
    print(f'\n▸ {col}')
    print(df_raw[col].value_counts().head(5).to_string())

## 10. Issues Flagged for Cleaning

Document findings here before handing off to Notebook 02.

In [ ]:
# ── Extraction summary log ────────────────────────────────────────────────────
print('=' * 60)
print('  EXTRACTION SUMMARY — ISSUES TO ADDRESS IN NOTEBOOK 02')
print('=' * 60)

print(f"""
Dataset     : {RAW_DATA_PATH}
Shape       : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns

Key Issues Identified
─────────────────────────────────────────────────────────
1. MISSING VALUES
   • 'weight'           — high missingness (≈97%); evaluate drop vs impute
   • 'payer_code'       — moderate missingness; impute or mark as 'Unknown'
   • 'medical_specialty'— moderate missingness; impute or mark as 'Unknown'
   • 'race'             — small % missing; impute with mode or 'Unknown'

2. ENCODING / CODING
   • admission_type_id, discharge_disposition_id, admission_source_id
     are integer codes — map to human-readable labels
   • diag_1/2/3 use ICD-9 codes — group into disease categories
   • age is stored as bracket strings (e.g. '[70-80)') — convert to midpoint

3. DUPLICATES / SCOPE
   • Multiple encounters per patient_nbr — keep all (encounter-level analysis)
   • Deceased patients (discharge_disposition_id 11, 19, 20, 21)
     typically excluded from readmission studies — flag for removal
   • Hospice discharges — review for exclusion

4. MEDICATION COLUMNS
   • 23 medication columns with values: No / Steady / Up / Down
   • Need ordinal encoding for statistical analysis

5. TARGET VARIABLE
   • 'readmitted': values '<30', '>30', 'NO'
   • Consider binary version: readmitted_30day (1 if '<30', else 0)

─────────────────────────────────────────────────────────
→ Hand off to: notebooks/02_cleaning.ipynb
""")

print('✅ Extraction notebook complete. Raw data committed to data/raw/. DO NOT EDIT.')